In [28]:
import sys

# Increase recursion limit to prevent potential issues
sys.setrecursionlimit(100000)

In [ ]:
# Step 2: Import necessary libraries
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam
import os
import warnings

# Suppress all Python warnings
warnings.filterwarnings('ignore')

# Set TensorFlow log level to suppress warnings and info messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0 = all logs, 1 = filter out INFO, 2 = filter out INFO and WARNING, 3 = ERROR only



In [30]:
# Step 3: Load and preprocess the MNIST dataset
(x_train, y_train), (x_val, y_val) = mnist.load_data()
x_train, x_val = x_train / 255.0, x_val / 255.0

print(f'Training data shape: {x_train.shape}')
print(f'Validation data shape: {x_val.shape}')

Training data shape: (60000, 28, 28)
Validation data shape: (10000, 28, 28)


In [31]:
def build_model(hp):
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [32]:
# Create a RandomSearch Tuner 

tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='intro_to_kt'
)

# Display a summary of the search space 
tuner.search_space_summary()

Reloading Tuner from my_dir/intro_to_kt/tuner0.json
Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


In [33]:
# Run the hyperparameter search 
tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val)) 

# Display a summary of the results 
tuner.results_summary() 

Results summary
Results in my_dir/intro_to_kt
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 09 summary
Hyperparameters:
units: 416
learning_rate: 0.0010848539808730223
Score: 0.9794500172138214

Trial 06 summary
Hyperparameters:
units: 448
learning_rate: 0.0005525349394079942
Score: 0.9794500172138214

Trial 07 summary
Hyperparameters:
units: 352
learning_rate: 0.0011025533633580264
Score: 0.979200005531311

Trial 04 summary
Hyperparameters:
units: 288
learning_rate: 0.0005452907696240787
Score: 0.9787999987602234

Trial 05 summary
Hyperparameters:
units: 320
learning_rate: 0.0019343087506326587
Score: 0.9779499769210815

Trial 08 summary
Hyperparameters:
units: 256
learning_rate: 0.00017981225010431958
Score: 0.9693500101566315

Trial 03 summary
Hyperparameters:
units: 384
learning_rate: 0.007622091714975342
Score: 0.9664500057697296

Trial 02 summary
Hyperparameters:
units: 96
learning_rate: 0.00985793043338185
Score: 0.9607999920845032

Trial 01 summa

In [34]:
# Step 1: Retrieve the best hyperparameters 

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0] 
print(f""" 

The optimal number of units in the first dense layer is {best_hps.get('units')}. 

The optimal learning rate for the optimizer is {best_hps.get('learning_rate')}. 

""") 

# Step 2: Build and Train the Model with Best Hyperparameters 
model = tuner.hypermodel.build(best_hps) 
model.fit(x_train, y_train, epochs=10, validation_split=0.2) 

# Evaluate the model on the test set 
test_loss, test_acc = model.evaluate(x_val, y_val) 
print(f'Test accuracy: {test_acc}') 

 

The optimal number of units in the first dense layer is 416. 

The optimal learning rate for the optimizer is 0.0010848539808730223. 


Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 782us/step - accuracy: 0.9325 - loss: 0.2286 - val_accuracy: 0.9658 - val_loss: 0.1143
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 783us/step - accuracy: 0.9725 - loss: 0.0918 - val_accuracy: 0.9704 - val_loss: 0.0923
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 796us/step - accuracy: 0.9813 - loss: 0.0607 - val_accuracy: 0.9719 - val_loss: 0.0873
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 854us/step - accuracy: 0.9873 - loss: 0.0411 - val_accuracy: 0.9737 - val_loss: 0.0913
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 793us/step - accuracy: 0.9905 - loss: 0.0292 - val_accuracy: 0.9753 - val_loss: 0.0940
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 794us/step - accuracy: 0.9923 - loss: 0.0235 - val_accuracy: 0.9749 - val_loss: 0.0868
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 791us/step - accuracy: 0

In [35]:
import keras_tuner as kt 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 
from tensorflow.keras.datasets import mnist 
from tensorflow.keras.optimizers import Adam 

# Step 3: Load and preprocess the MNIST data set 
(x_train, y_train), (x_val, y_val) = mnist.load_data() 
x_train, x_val = x_train / 255.0, x_val / 255.0 

# Print the shapes of the training and validation datasets
print(f'Training data shape: {x_train.shape}') 
print(f'Validation data shape: {x_val.shape}')

Training data shape: (60000, 28, 28)
Validation data shape: (10000, 28, 28)


In [36]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
import keras_tuner as kt

# Step 1: Define a model-building function
def build_model(hp):
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [37]:
import keras_tuner as kt

# Step 1: Create a RandomSearch Tuner
tuner = kt.RandomSearch(
    build_model,  # Ensure 'build_model' function is defined from previous code
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='intro_to_kt'
)

# Display a summary of the search space
tuner.search_space_summary()

Reloading Tuner from my_dir/intro_to_kt/tuner0.json
Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


In [38]:
# Run the hyperparameter search 
tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val)) 

# Display a summary of the results 
tuner.results_summary() 

Results summary
Results in my_dir/intro_to_kt
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 06 summary
Hyperparameters:
units: 448
learning_rate: 0.0005525349394079942
Score: 0.9794500172138214

Trial 09 summary
Hyperparameters:
units: 416
learning_rate: 0.0010848539808730223
Score: 0.9794500172138214

Trial 07 summary
Hyperparameters:
units: 352
learning_rate: 0.0011025533633580264
Score: 0.979200005531311

Trial 04 summary
Hyperparameters:
units: 288
learning_rate: 0.0005452907696240787
Score: 0.9787999987602234

Trial 05 summary
Hyperparameters:
units: 320
learning_rate: 0.0019343087506326587
Score: 0.9779499769210815

Trial 08 summary
Hyperparameters:
units: 256
learning_rate: 0.00017981225010431958
Score: 0.9693500101566315

Trial 03 summary
Hyperparameters:
units: 384
learning_rate: 0.007622091714975342
Score: 0.9664500057697296

Trial 02 summary
Hyperparameters:
units: 96
learning_rate: 0.00985793043338185
Score: 0.9607999920845032

Trial 01 summa

In [39]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0] 

print(f""" 

The optimal number of units in the first dense layer is {best_hps.get('units')}. 

The optimal learning rate for the optimizer is {best_hps.get('learning_rate')}. 

""") 

 # Step 2: Build and train the model with best hyperparameters 

model = tuner.hypermodel.build(best_hps) 

model.fit(x_train, y_train, epochs=10, validation_split=0.2) 

 # Evaluate the model on the validation set 

val_loss, val_acc = model.evaluate(x_val, y_val) 

print(f'Validation accuracy: {val_acc}')

 

The optimal number of units in the first dense layer is 448. 

The optimal learning rate for the optimizer is 0.0005525349394079942. 


Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 829us/step - accuracy: 0.9237 - loss: 0.2700 - val_accuracy: 0.9560 - val_loss: 0.1501
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 832us/step - accuracy: 0.9676 - loss: 0.1122 - val_accuracy: 0.9680 - val_loss: 0.1076
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 800us/step - accuracy: 0.9788 - loss: 0.0718 - val_accuracy: 0.9745 - val_loss: 0.0868
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 806us/step - accuracy: 0.9856 - loss: 0.0499 - val_accuracy: 0.9776 - val_loss: 0.0780
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 793us/step - accuracy: 0.9898 - loss: 0.0352 - val_accuracy: 0.9772 - val_loss: 0.0766
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 797us/step - accuracy: 0.9922 - loss: 0.0263 - val_accuracy: 0.9732 - val_loss: 0.0932
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 810us/step - accuracy: 0